# Suite CPP — The Ccpp language

C and C++ as one tree language (`@mbse/programs/Ccpp/Syntax`): its kinds and categories, organized as the
standard's grammar is, the identifiers it accepts, and where each kind and feature exists, which the C++ standards
check.

In [ ]:
import { Ccpp17, Ccpp20, Syntax as S } from "@mbse/programs/Ccpp";
import { CcppStandard } from "@mbse/programs/Ccpp/_Standard";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const [C23, CPP23, CPP26] = [new CcppStandard(2023, "C"), new CcppStandard(2023), new CcppStandard(2026)];
const name = (spelling: unknown) => new S.Identifier({ spelling });
const ident = (spelling: string) => new S.IdExpression({ name: name(spelling) });
const one = () => new S.IntegerLiteral({ spelling: "1" });

## CPP-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
const kinds = L.kinds();
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
equal([...L.categories().keys()], ["Name", "Expression", "Literal", "Capture", "Requirement", "Designator", "Statement",
  "Declaration", "Specifier", "Declarator", "Parameter", "Initializer", "ExceptionSpecification", "ContractSpecifier",
  "AttributeSpecifier", "TemplateParameter", "Directive"]);
assert(new S.IntegerLiteral() instanceof S.Literal && new S.IntegerLiteral() instanceof S.Expression);
assert(S.KINDS.every((k) => k.NAME === `Programs.Ccpp.${k.KIND}`));
const grammar = L.grammar() as any;
equal(grammar.base, { "C++": 1998, C: 1989 });
assert(grammar.kinds.length === S.KINDS.length);
equal(grammar.categories.find((c: any) => c.name === "Literal"), { name: "Literal", base: "Expression" });
equal([S.cpp(2011), S.c(1999), S.both(2011, 2023)], [{ "C++": 2011 }, { C: 1999 }, { "C++": 2011, C: 2023 }]);
assert(new S.Comment({ text: " x" }).block === false && new S.TranslationUnit().items.length === 0);
console.log("ok");

## CPP-02 · Identifiers are letters, digits, `_` and `$`, not starting with a digit

In [ ]:
equal(L.validate(name("x_1$")), []);
equal(L.validate(name("Größe")), []);
equal(L.validate(name("1x")), ["'1x' is not an identifier"]);
equal(L.validate(name("a b")), ["'a b' is not an identifier"]);
equal(L.validate(name("")), []);
equal(L.validate(new S.Identifier()), ["an Identifier needs a spelling"]);
equal(L.validate(name(3n)), ["Identifier.spelling must be a str, got int"]);
console.log("ok");

## CPP-03 · Where kinds exist: C++ only, C only, both, and since which standard

In [ ]:
const lambda = new S.LambdaExpression({ body: new S.CompoundStatement() });
equal(Ccpp17.check(lambda), []);
equal(C23.check(lambda), ["LambdaExpression is not C"]);
const generic = new S.GenericSelection({ controlling: one(), associations: [new S.GenericAssociation({ value: one() })] });
equal(C23.check(generic), []);
equal(Ccpp20.check(generic), ["GenericSelection is not C++", "associations[0]: GenericAssociation is not C++"]);
const requires = new S.RequiresExpression({ requirements: [new S.SimpleRequirement({ expression: one() })] });
equal(Ccpp17.check(requires), ["RequiresExpression needs C++20", "requirements[0]: SimpleRequirement needs C++20"]);
equal(Ccpp20.check(requires), []);
const throws = new S.ThrowSpecifier({ types: [new S.TypeId({ specifiers: [new S.PrimitiveTypeSpecifier({ keyword: "int" })] })] });
equal(Ccpp17.check(throws), ["ThrowSpecifier with types was removed in C++17"]);
equal(Ccpp17.check(new S.ThrowSpecifier()), []);
equal(Ccpp20.check(new S.ThrowSpecifier()), ["ThrowSpecifier was removed in C++20"]);
equal(Ccpp17.check(new S.DeclSpecifier({ keyword: "register" })), ["DeclSpecifier.keyword 'register' was removed in C++17"]);
equal(C23.check(new S.DeclSpecifier({ keyword: "register" })), []);
equal(Ccpp20.check(new S.DeclSpecifier({ keyword: "__inline" })), []);
equal(Ccpp20.check(new S.StatementExpression({ body: new S.CompoundStatement() })), []); // an extension
equal(Ccpp17.check(new S.BinaryExpression({ left: one(), operator: "<=>", right: one() })),
  ["BinaryExpression.operator '<=>' needs C++20"]);
equal(Ccpp17.check(new S.IfStatement({ constexpr: true, consequence: new S.CompoundStatement() })), []);
equal(Ccpp20.check(new S.IfStatement({ consteval: true, consequence: new S.CompoundStatement() })),
  ["IfStatement.consteval needs C++23"]);
console.log("ok");

## CPP-04 · Kinds whose features depend on their fields

In [ ]:
const subscript = new S.SubscriptExpression({ object: ident("m"), indices: [one(), one()] });
equal(Ccpp20.check(subscript), ["SubscriptExpression with several indices needs C++23"]);
equal(CPP23.check(subscript), []);
equal(Ccpp20.check(new S.SubscriptExpression({ object: ident("m"), indices: [one()] })), []);
const field = new S.FieldDesignator({ name: name("x") });
const designated = new S.DesignatedInitializer({ designators: [field, new S.IndexDesignator({ index: one() })],
  initializer: new S.EqualInitializer({ value: one() }) });
equal(Ccpp20.check(designated), ["DesignatedInitializer with array or nested designators is not C++",
  "designators[1]: IndexDesignator is not C++"]);
equal(C23.check(designated), []);
equal(Ccpp17.check(new S.DesignatedInitializer({ designators: [field], initializer: new S.EqualInitializer({ value: one() }) })),
  ["DesignatedInitializer needs C++20"]);
for (const label of [new S.LabeledStatement({ label: name("l") }), new S.CaseStatement({ value: one() }), new S.DefaultStatement()]) {
  equal(Ccpp20.check(label), [`${label.kind().KIND} without a statement needs C++23`]);
  label.statement = new S.BreakStatement();
  equal(Ccpp20.check(label), []);
}
equal(C23.check(new S.TypeofSpecifier({ keyword: "typeof", operand: one() })), []);
equal(Ccpp20.check(new S.TypeofSpecifier({ keyword: "typeof", operand: one() })), ["TypeofSpecifier is not C++"]);
equal(Ccpp20.check(new S.TypeofSpecifier({ keyword: "__typeof__", operand: one() })), []);
const bindings = new S.StructuredBindingDeclarator({ bindings: [new S.IdDeclarator({ name: name("a") })] });
equal(Ccpp17.check(bindings), []);
bindings.bindings.push(new S.PackDeclarator({ declarator: new S.IdDeclarator({ name: name("r") }) }),
  new S.IdDeclarator({ name: name("b"), attributes: [new S.StandardAttributeSpecifier()] }));
equal(Ccpp20.check(bindings), ["StructuredBindingDeclarator with a pack needs C++26",
  "StructuredBindingDeclarator with attributes needs C++26"]);
equal(CPP26.check(bindings), []);
const nested = new S.NamespaceDefinition({ names: [new S.NamespaceName({ name: name("a") }), new S.NamespaceName({ name: name("b") })] });
equal(Ccpp17.check(nested), []);
equal(Ccpp17.check(new S.NamespaceDefinition({ names: [new S.NamespaceName({ name: name("a") })] })), []);
(nested.names[1] as S.NamespaceName).inline = true;
equal(Ccpp17.check(nested), ["NamespaceDefinition with inline nested names needs C++20"]);
assert(new CcppStandard(2014).check(nested)[0] === "NamespaceDefinition with nested names needs C++17");
const declarator = new S.UsingDeclarator({ name: name("a") });
const using = new S.UsingDeclaration({ declarators: [declarator, declarator] });
equal(new CcppStandard(2014).check(using), ["UsingDeclaration with several declarators needs C++17"]);
equal(Ccpp17.check(using), []);
const assertion = new S.StaticAssertDeclaration({ keyword: "static_assert", condition: one() });
equal(new CcppStandard(2014).check(assertion), ["StaticAssertDeclaration without a message needs C++17"]);
assertion.message = new S.StringLiteral({ text: "m" });
equal(new CcppStandard(2014).check(assertion), []);
equal(Ccpp20.check(new S.OtherDirective({ directive: "warning" })), ["#warning needs C++23"]);
equal(Ccpp20.check(new S.OtherDirective({ directive: "embed" })), ["#embed needs C++26"]);
equal(Ccpp17.check(new S.OtherDirective({ directive: "pragma" })), []);
equal(Ccpp17.check(new S.OtherDirective()), []);
console.log("ok");